In [ ]:
## Aula 1: Cosmology 101 & Power spectrum
# Isabela, Louis, Bruno

In [ ]:
# Importing python packages

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from classy import Class
from time import perf_counter

rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 14
rcParams['xtick.labelsize'] = 14
rcParams['ytick.labelsize'] = 14

## Espectro de potência da matéria

### 1. Qual o efeito de $\sigma_8$?

In [ ]:
# ----------------------------------------------------------
# Lista de valores de sigma8 a serem testados
# ----------------------------------------------------------
sig8_list = [0.75, 0.80, 0.85, 0.90, 0.95]

# ----------------------------------------------------------
# Parâmetros cosmológicos de referência
# ----------------------------------------------------------
h_ref   = 0.67
Ob_ref  = 0.049
Om_ref  = 0.315
ns_ref  = 0.965
P_k_max_h_Mpc = 5.0
k_per_decade  = 50

# ----------------------------------------------------------
# Cria dicionário de cosmologias (variando apenas sigma8)
# ----------------------------------------------------------
cosmo_dict = {}
for sig8_ref in sig8_list:
    cosmo_name = "cosmo_sig8_{:.2f}".format(sig8_ref)
    cosmo_dict[cosmo_name] = {
        'output': 'mPk',
        # 'radiation_streaming_approximation': 3,  # Desativar aprox. de fluido p/ radiação
        # 'ncdm_fluid_approximation': 3,           # Desativar aprox. de fluido p/ ν massivos
        # 'ur_fluid_approximation': 2,             # Desativar aprox. de fluido p/ ν sem massa
        'Omega_Lambda': 0,                        # Ativa automaticamente a evolução _fld (DE dinâmica)
        'w0_fld': '-1.',                          # Equação de estado da DE (constante)
        'wa_fld': '0.0',                          # Variação temporal de w(a)
        'sigma8': sig8_ref,                       # Variável principal testada
        'n_s': ns_ref,                            # Índice espectral primordial
        'background_verbose': 0,                  # Verbosidade do módulo background
        'perturbations_verbose': 0,               # Verbosidade do módulo perturbations
        'gauge': 'Synchronous',                   # Gauge síncrono (mais estável num.)
        'z_pk': '1.0, 0.0',                       # Redshifts para saída de P(k)
        'P_k_max_h/Mpc': P_k_max_h_Mpc,           # k máximo para P(k)
        'h': h_ref,                               # Parâmetro de Hubble reduzido
        'Omega_b': Ob_ref,                        # Densidade de bárions
        'Omega_cdm': Om_ref - Ob_ref              # Densidade de CDM
    }

# ----------------------------------------------------------
# Loop: calcular cada cosmologia, medir tempo e armazenar
# ----------------------------------------------------------
results = {}          # guarda os objetos Class
durations = {}        # guarda tempos em segundos por cosmologia

for name, params in cosmo_dict.items():
    start = perf_counter()
    cosmo = Class()
    cosmo.set(params)
    cosmo.compute()
    end = perf_counter()

    results[name] = cosmo
    durations[name] = end - start
    print("{} resolvida (σ₈ = {:.2f}) em {:.2f} s".format(name, params['sigma8'], durations[name]))

# ----------------------------------------------------------
# Sumário final (ordenado do mais rápido ao mais lento)
# ----------------------------------------------------------
print("\n== Tempo por cosmologia ==")
for name in sorted(durations, key=durations.get):
    print("{:<18s}  {:6.2f} s".format(name, durations[name]))

# Exemplo de acesso a P(k, z=0):
# pk = results['cosmo_sig8_0.85'].pk(k=0.1, z=0)